<h1 align="center">Neural Network Lab</h1>

<img src="image/notmnist.png">
In this lab, you'll build a neural network with PyTorch to label images of English letters! The data you are using, <a href="http://yaroslavvb.blogspot.com/2011/09/notmnist-dataset.html">notMNIST</a>, consists of images of a letter from A to J in different fonts.

The above images are a few examples of the data you'll be training on. After training the network, you will compare your prediction model against test data. Your goal, by the end of this lab, is to make predictions against that test set with at least an 80% accuracy. Let's jump in!

To start this lab, you first need to import all the necessary modules. Run the code below. If it runs successfully, it will print "`All modules imported`".

In [ ]:
import hashlib
import math
import os
from urllib.request import urlretrieve
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer
from tqdm import tqdm

# Use the GPU when there is one; otherwise fall back to the CPU.
device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available()
                      else "cpu")
print("Using device:", device)

# Set the environment variable DL_SMOKE=1 to use a much smaller subset of the data
# (handy for a quick check that everything runs).
SMOKE = os.environ.get('DL_SMOKE') == '1'

print('All modules imported.')

The notMNIST dataset is too large for many computers to handle.  It contains 210,000 images for just training.  You'll be using a subset of this data, 150,000 randomly chosen images.

In [ ]:
def download(url, file):
    """
    Download file from <url>
    :param url: URL to file
    :param file: Local file path
    """
    if not os.path.isfile(file):
        print('Downloading ' + file + '...')
        urlretrieve(url, file)
        print('Download Finished')

# Download the training and test dataset.
download('https://s3.amazonaws.com/udacity-sdc/notMNIST_train.zip', 'notMNIST_train.zip')
download('https://s3.amazonaws.com/udacity-sdc/notMNIST_test.zip', 'notMNIST_test.zip')

# Make sure the files aren't corrupted
assert hashlib.md5(open('notMNIST_train.zip', 'rb').read()).hexdigest() == 'c8673b3f28f489e9cdf3a3d74e2ac8fa',\
        'notMNIST_train.zip file is corrupted.  Remove the file and try again.'
assert hashlib.md5(open('notMNIST_test.zip', 'rb').read()).hexdigest() == '5d3c7e653e63471c88df796156a9dfa9',\
        'notMNIST_test.zip file is corrupted.  Remove the file and try again.'

# Wait until you see that all files have been downloaded.
print('All files downloaded.')

In [ ]:
def uncompress_features_labels(file, n_samples=None):
    """
    Uncompress features and labels from a zip file
    :param file: The zip file to extract the data from
    :param n_samples: If given, only read this many randomly chosen images
    """
    features = []
    labels = []

    with ZipFile(file) as zipf:
        # Skip the directory entries
        filenames = [name for name in zipf.namelist() if not name.endswith('/')]
        if n_samples is not None and n_samples < len(filenames):
            filenames = list(np.random.RandomState(0).choice(filenames, n_samples, replace=False))

        # Get features and labels from all files (tqdm shows a progress bar)
        for filename in tqdm(filenames, unit='files'):
            with zipf.open(filename) as image_file:
                image = Image.open(image_file)
                image.load()
                # Load image data as 1 dimensional array
                # We're using float32 to save on memory space
                feature = np.array(image, dtype=np.float32).flatten()

            # Get the the letter from the filename.  This is the letter of the image.
            label = os.path.split(filename)[1][0]

            features.append(feature)
            labels.append(label)
    return np.array(features), np.array(labels)

# Get the features and labels from the zip files
train_size = 40000 if SMOKE else 150000
test_size = 5000 if SMOKE else None
train_features, train_labels = uncompress_features_labels('notMNIST_train.zip', train_size)
test_features, test_labels = uncompress_features_labels('notMNIST_test.zip', test_size)

# Set flags for feature engineering.  This will prevent you from skipping an important step.
is_features_normal = False
is_labels_encod = False

# Wait until you see that all features and labels have been uncompressed.
print('All features and labels uncompressed.')

<img src="image/Mean_Variance_Image.png" style="height: 75%;width: 75%; position: relative; right: 5%">
## Problem 1
The first problem involves normalizing the features for your training and test data.

Implement Min-Max scaling in the `normalize_grayscale()` function to a range of `a=0.1` and `b=0.9`. After scaling, the values of the pixels in the input data should range from 0.1 to 0.9.

Since the raw notMNIST image data is in [grayscale](https://en.wikipedia.org/wiki/Grayscale), the current values range from a min of 0 to a max of 255.

Min-Max Scaling:
$
X'=a+{\frac {\left(X-X_{\min }\right)\left(b-a\right)}{X_{\max }-X_{\min }}}
$

*If you're having trouble solving problem 1, you can view the solution in `NotMNIST_Lab_PyTorch_Solution.ipynb`.*

In [ ]:
# Problem 1 - Implement Min-Max scaling for grayscale image data
def normalize_grayscale(image_data):
    """
    Normalize the image data with Min-Max scaling to a range of [0.1, 0.9]
    :param image_data: The image data to be normalized
    :return: Normalized image data
    """
    a, b = 0.1, 0.9
    grayscale_min, grayscale_max = 0, 255
    return a + (image_data - grayscale_min) * (b - a) / (grayscale_max - grayscale_min)


### DON'T MODIFY ANYTHING BELOW ###
# Test Cases
np.testing.assert_array_almost_equal(
    normalize_grayscale(np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 255])),
    [0.1, 0.103137254902, 0.106274509804, 0.109411764706, 0.112549019608, 0.11568627451, 0.118823529412, 0.121960784314,
     0.125098039216, 0.128235294118, 0.13137254902, 0.9],
    decimal=3)
np.testing.assert_array_almost_equal(
    normalize_grayscale(np.array([0, 1, 10, 20, 30, 40, 233, 244, 254,255])),
    [0.1, 0.103137254902, 0.13137254902, 0.162745098039, 0.194117647059, 0.225490196078, 0.830980392157, 0.865490196078,
     0.896862745098, 0.9])

if not is_features_normal:
    train_features = normalize_grayscale(train_features)
    test_features = normalize_grayscale(test_features)
    is_features_normal = True

print('Tests Passed!')

In [ ]:
if not is_labels_encod:
    # Turn labels into numbers and apply One-Hot Encoding
    encoder = LabelBinarizer()
    encoder.fit(train_labels)
    train_labels = encoder.transform(train_labels)
    test_labels = encoder.transform(test_labels)

    # Change to float32, so it can be multiplied against the features in PyTorch, which are float32
    train_labels = train_labels.astype(np.float32)
    test_labels = test_labels.astype(np.float32)
    is_labels_encod = True

print('Labels One-Hot Encoded')

In [ ]:
assert is_features_normal, 'You skipped the step to normalize the features'
assert is_labels_encod, 'You skipped the step to One-Hot Encode the labels'

# Get randomized datasets for training and validation
train_features, valid_features, train_labels, valid_labels = train_test_split(
    train_features,
    train_labels,
    test_size=0.05,
    random_state=832289)

print('Training features and labels randomized and split.')

## Problem 2

Now it's time to build a simple neural network using PyTorch. Here, your network will be just an input layer and an output layer.

<img src="image/network_diagram.png" style="height: 40%;width: 40%; position: relative; right: 10%">

For the input here the images have been flattened into a vector of $28 \times 28 = 784$ features. Then, we're trying to predict the image letter so there are 10 output units, one for each label. Of course, feel free to add hidden layers if you want, but this notebook is built to guide you through a single layer network.

The data (`features` and `labels`) is converted to `float32` tensors for you below. For the network itself you need the following `float32` tensors, which PyTorch will train using gradients:
 - `weights`
  - A tensor of shape `(features_count, labels_count)` with random numbers from a truncated normal distribution, created with `requires_grad=True` so PyTorch tracks gradients for it.
    - See [`torch.nn.init.trunc_normal_`](https://pytorch.org/docs/stable/nn.init.html#torch.nn.init.trunc_normal_) for help. Create the tensor with `torch.empty(...)`, fill it with `trunc_normal_`, then call `.requires_grad_()`.
 - `biases`
  - A tensor of shape `(labels_count,)` with all zeros, also with `requires_grad=True`.
    - See [`torch.zeros`](https://pytorch.org/docs/stable/generated/torch.zeros.html) for help.

*If you're having trouble solving problem 2, review the linear function section of the class. If that doesn't help, the solution is in `NotMNIST_Lab_PyTorch_Solution.ipynb`.*

In [ ]:
# All the pixels in the image (28 * 28 = 784)
features_count = 784
# All the labels
labels_count = 10

# Problem 2 - Set the weights and biases tensors
weights = torch.nn.init.trunc_normal_(torch.empty(features_count, labels_count, device=device)).requires_grad_()
biases = torch.zeros(labels_count, device=device, requires_grad=True)


### DON'T MODIFY ANYTHING BELOW ###

# Test Cases
assert isinstance(weights, torch.Tensor) and weights.requires_grad, 'weights must be a tensor with requires_grad=True'
assert isinstance(biases, torch.Tensor) and biases.requires_grad, 'biases must be a tensor with requires_grad=True'
assert weights.is_leaf and biases.is_leaf, 'Create weights and biases directly on the device, do not move them afterwards'

assert tuple(weights.shape) == (784, 10), 'The shape of weights is incorrect'
assert tuple(biases.shape) == (10,), 'The shape of biases is incorrect'

assert weights.dtype == torch.float32, 'weights must be type float32'
assert biases.dtype == torch.float32, 'biases must be type float32'

assert weights.std().item() > 0, 'weights must be random numbers, not constants'
assert not np.count_nonzero(biases.detach().cpu().numpy()), 'biases must be zeros'

# Remember the starting values so we can train the same network again later
initial_weights = weights.detach().clone()

# The data as tensors on the active device
train_X, train_Y = torch.tensor(train_features, device=device), torch.tensor(train_labels, device=device)
valid_X, valid_Y = torch.tensor(valid_features, device=device), torch.tensor(valid_labels, device=device)
test_X, test_Y = torch.tensor(test_features, device=device), torch.tensor(test_labels, device=device)

# Linear Function WX + b
def logits_fn(features):
    return features @ weights + biases

# Cross entropy between the one-hot labels and the softmax of the logits
def cross_entropy_loss(logits, labels):
    return -(labels * torch.log_softmax(logits, dim=1)).sum(dim=1).mean()

# Test Cases
for X, Y in [(train_X, train_Y), (valid_X, valid_Y), (test_X, test_Y)]:
    cross_entropy_loss(logits_fn(X), Y)

print('Tests Passed!')

In [ ]:
# Calculate the accuracy of the predictions: the fraction of images whose most likely letter is the right one
def accuracy(features, labels):
    with torch.no_grad():
        is_correct_prediction = logits_fn(features).argmax(dim=1) == labels.argmax(dim=1)
        return is_correct_prediction.float().mean().item()

print('Accuracy function created.')

<img src="image/Learn_Rate_Tune_Image.png" style="height: 70%;width: 70%">
## Problem 3
Below are 2 parameter configurations for training the neural network. In each configuration, one of the parameters has multiple options. For each configuration, choose the option that gives the best acccuracy.

Parameter configurations:

Configuration 1
* **Epochs:** 1
* **Learning Rate:**
  * 0.8
  * 0.5
  * 0.1
  * 0.05
  * 0.01

Configuration 2
* **Epochs:**
  * 1
  * 2
  * 3
  * 4
  * 5
* **Learning Rate:** 0.2

The code will print out a Loss and Accuracy graph, so you can see how well the neural network performed.

*If you're having trouble solving problem 3, you can view the solution in `NotMNIST_Lab_PyTorch_Solution.ipynb`.*

In [ ]:
# Change if you have memory restrictions
batch_size = 128

# Problem 3 - Find the best parameters for each configuration
epochs = 5
learning_rate = 0.2


### DON'T MODIFY ANYTHING BELOW ###
def train(epochs, learning_rate, log=True):
    """Train the network from its starting weights; returns the logged batches, losses and accuracies."""
    # Start every run from the same starting values
    with torch.no_grad():
        weights.copy_(initial_weights)
        biases.zero_()

    # Gradient Descent
    optimizer = torch.optim.SGD([weights, biases], lr=learning_rate)

    # Measurements use for graphing loss and accuracy
    log_batch_step = 50
    batches, loss_batch, train_acc_batch, valid_acc_batch = [], [], [], []

    batch_count = int(math.ceil(len(train_X) / batch_size))
    for epoch_i in range(epochs):

        # Progress bar
        batches_pbar = tqdm(range(batch_count), desc='Epoch {:>2}/{}'.format(epoch_i+1, epochs), unit='batches')

        # The training cycle
        for batch_i in batches_pbar:
            # Get a batch of training features and labels
            batch_start = batch_i*batch_size
            batch_features = train_X[batch_start:batch_start + batch_size]
            batch_labels = train_Y[batch_start:batch_start + batch_size]

            # Gradient descent step: zero the gradients, compute the loss, backpropagate, update the weights
            optimizer.zero_grad()
            loss = cross_entropy_loss(logits_fn(batch_features), batch_labels)
            loss.backward()
            optimizer.step()

            # Log every 50 batches
            if log and not batch_i % log_batch_step:
                previous_batch = batches[-1] if batches else 0
                batches.append(log_batch_step + previous_batch)
                loss_batch.append(loss.item())
                train_acc_batch.append(accuracy(train_X, train_Y))
                valid_acc_batch.append(accuracy(valid_X, valid_Y))

    return batches, loss_batch, train_acc_batch, valid_acc_batch

batches, loss_batch, train_acc_batch, valid_acc_batch = train(epochs, learning_rate)

loss_plot = plt.subplot(211)
loss_plot.set_title('Loss')
loss_plot.plot(batches, loss_batch, 'g')
loss_plot.set_xlim([batches[0], batches[-1]])
acc_plot = plt.subplot(212)
acc_plot.set_title('Accuracy')
acc_plot.plot(batches, train_acc_batch, 'r', label='Training Accuracy')
acc_plot.plot(batches, valid_acc_batch, 'x', label='Validation Accuracy')
acc_plot.set_ylim([0, 1.0])
acc_plot.set_xlim([batches[0], batches[-1]])
acc_plot.legend(loc=4)
plt.tight_layout()
plt.show()

validation_accuracy = accuracy(valid_X, valid_Y)
print('Validation accuracy at {}'.format(validation_accuracy))

## Test
You're going to test your model against your hold out dataset/testing data.  This will give you a good indicator of how well the model will do in the real world.  You should have a test accuracy of at least 80%.

In [ ]:
### DON'T MODIFY ANYTHING BELOW ###
# Train again from the starting weights, then measure the accuracy against the test set
train(epochs, learning_rate, log=False)
test_accuracy = accuracy(test_X, test_Y)

assert test_accuracy >= 0.80, 'Test accuracy at {}, should be equal to or greater than 0.80'.format(test_accuracy)
print('Nice Job! Test Accuracy is {}'.format(test_accuracy))

# Multiple layers
Good job!  You built a one layer PyTorch network!  However, you might want to build more than one layer.  This is deep learning after all!  Try adding a hidden layer: replace `weights` and `biases` with an `nn.Sequential` of `nn.Linear` layers and an activation, like in `Digit_Recognition_PyTorch.ipynb`.